# CargoGuard SENTINEL - YOLOv8 Training
## X-Ray Security Screening Model

This notebook trains a YOLOv8 model for detecting prohibited items in X-ray cargo scans.

**Dataset:** SIXray v3 (5 classes: Gun, Knife, Pliers, Scissors, Wrench)

**Target:** mAP@0.5 > 80%, Recall > 88%

In [ ]:
# Install dependencies
!pip install ultralytics -q
!pip install grad-cam -q

In [ ]:
# Verify environment
import torch
from ultralytics import YOLO

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('Running on CPU')

In [ ]:
# Setup dataset paths
import os
from pathlib import Path

# Option A: Kaggle Dataset (upload your dataset first)
DATASET_PATH = '/kaggle/input/sixray-v3'

# Option B: Local path (if running locally)
# DATASET_PATH = 'data/raw/sixray_v3'

# Create data.yaml for training
yaml_content = f'''
path: {DATASET_PATH}
train: train/images
val: valid/images
test: test/images

nc: 5
names:
  0: Gun
  1: Knife
  2: Pliers
  3: Scissors
  4: Wrench
'''

with open('/kaggle/working/data.yaml', 'w') as f:
    f.write(yaml_content)

print('data.yaml created')
print(yaml_content)

In [ ]:
# Verify dataset
import os

for split in ['train', 'valid', 'test']:
    img_dir = f'{DATASET_PATH}/{split}/images'
    lbl_dir = f'{DATASET_PATH}/{split}/labels'
    
    if os.path.exists(img_dir):
        num_imgs = len([f for f in os.listdir(img_dir) if f.endswith('.jpg')])
        num_lbls = len([f for f in os.listdir(lbl_dir) if f.endswith('.txt')])
        print(f'{split}: {num_imgs} images, {num_lbls} labels')
    else:
        print(f'{split}: NOT FOUND')

## Train YOLOv8 Model

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8-small (downloads automatically ~22MB)
model = YOLO('yolov8s.pt')

# Fine-tune on X-ray security dataset
results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0 if torch.cuda.is_available() else 'cpu',
    
    # Optimizer settings
    optimizer='SGD',
    lr0=0.01,
    momentum=0.9,
    weight_decay=0.0005,
    
    # Augmentation - tuned for X-ray images
    augment=True,
    mosaic=1.0,
    mixup=0.1,
    copy_paste=0.1,
    flipud=0.0,        # X-rays should not be flipped vertically
    fliplr=0.5,
    degrees=0.0,       # No rotation for X-rays
    translate=0.1,
    scale=0.5,
    hsv_h=0.0,         # X-rays are grayscale
    hsv_s=0.0,
    hsv_v=0.4,         # Only brightness variation
    
    # Training config
    patience=15,       # Early stopping
    save=True,
    save_period=10,    # Save checkpoint every 10 epochs
    project='/kaggle/working/runs',
    name='cargoguard_v1',
    exist_ok=True,
    pretrained=True,
    verbose=True
)

print(f'\nBest mAP50: {results.results_dict["metrics/mAP50(B)"]:.3f}')
print(f'Training complete. Model saved to: /kaggle/working/runs/cargoguard_v1/weights/best.pt')

## Monitor Training Progress

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load training results
results_csv = '/kaggle/working/runs/cargoguard_v1/results.csv'

try:
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    axes[0,0].plot(df['epoch'], df['metrics/mAP50(B)'], 'b-')
    axes[0,0].set_title('mAP@0.5')
    axes[0,0].set_xlabel('Epoch')
    axes[0,0].grid(True)

    axes[0,1].plot(df['epoch'], df['metrics/precision(B)'], 'g-')
    axes[0,1].set_title('Precision')
    axes[0,1].set_xlabel('Epoch')
    axes[0,1].grid(True)

    axes[1,0].plot(df['epoch'], df['metrics/recall(B)'], 'r-')
    axes[1,0].set_title('Recall')
    axes[1,0].set_xlabel('Epoch')
    axes[1,0].grid(True)

    axes[1,1].plot(df['epoch'], df['train/box_loss'], 'k-', label='train')
    axes[1,1].plot(df['epoch'], df['val/box_loss'], 'b--', label='val')
    axes[1,1].set_title('Box Loss')
    axes[1,1].legend()
    axes[1,1].grid(True)

    plt.tight_layout()
    plt.savefig('/kaggle/working/training_curves.png', dpi=150)
    plt.show()
except Exception as e:
    print(f'Could not load results: {e}')

## Evaluate on Test Set

In [ ]:
from ultralytics import YOLO

# Load best model
model = YOLO('/kaggle/working/runs/cargoguard_v1/weights/best.pt')

# Run validation on test split
metrics = model.val(
    data='/kaggle/working/data.yaml',
    split='test',
    imgsz=640,
    conf=0.25,
    iou=0.45,
    save_json=True,
    plots=True
)

print('=' * 50)
print('TEST SET RESULTS')
print('=' * 50)
print(f'mAP@0.5:      {metrics.box.map50:.4f}')
print(f'mAP@0.5:0.95: {metrics.box.map:.4f}')
print(f'Precision:    {metrics.box.mp:.4f}')
print(f'Recall:       {metrics.box.mr:.4f}')
print()
print('Per-class AP@0.5:')
class_names = ['Gun', 'Knife', 'Pliers', 'Scissors', 'Wrench']
for name, ap in zip(class_names, metrics.box.ap50):
    print(f'  {name:12s}: {ap:.4f}')

## Test Inference

In [ ]:
import cv2
import matplotlib.pyplot as plt
from pathlib import Path

# Load model
model = YOLO('/kaggle/working/runs/cargoguard_v1/weights/best.pt')

# Find test images
test_dir = Path(f'{DATASET_PATH}/test/images')
test_images = list(test_dir.glob('*.jpg'))[:6]

# Run inference
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, img_path in enumerate(test_images):
    results = model.predict(source=str(img_path), conf=0.25, verbose=False)
    annotated = results[0].plot()
    axes[i].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
    axes[i].set_title(img_path.name[:20])
    axes[i].axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/test_predictions.png', dpi=150)
plt.show()

## Export Model

In [ ]:
# Export to ONNX for faster CPU inference
model = YOLO('/kaggle/working/runs/cargoguard_v1/weights/best.pt')
model.export(format='onnx', imgsz=640, opset=12, simplify=True)

print('ONNX export complete!')
print('Download best.pt and best.onnx from /kaggle/working/runs/cargoguard_v1/weights/')

In [ ]:
# Summary
print('\n' + '=' * 60)
print('TRAINING COMPLETE - CargoGuard SENTINEL')
print('=' * 60)
print()
print('Files to download:')
print('  1. /kaggle/working/runs/cargoguard_v1/weights/best.pt')
print('  2. /kaggle/working/runs/cargoguard_v1/weights/best.onnx')
print('  3. /kaggle/working/training_curves.png')
print('  4. /kaggle/working/test_predictions.png')
print()
print('Copy best.pt to: cargoguard/models/yolo/best.pt')